**Aliasing with an unknown waveform.**
In practice a signal's frequency content is often not known in advance.
This notebook explores what happens when you sample a composite waveform
at various rates without knowing how many frequency components it contains
or how high they reach.

The "unknown" signal is a product of four sinusoids:

$$
f(x) = \sin(4\pi x)\cdot\sin(6\pi x)\cdot\sin(10\pi x)\cdot\sin(14\pi x)
$$

Its component frequencies are 2, 3, 5, and 7 cycles/unit.
Expanding the product using $\sin A \sin B = \tfrac{1}{2}[\cos(A-B)-\cos(A+B)]$
reveals the true spectral content - but the $f=3$ and $f=7$ terms cancel,
leaving only four frequency components at $f = 1, 11, 13, 17$ cycles/unit,
each with equal amplitude $\tfrac{1}{8}$.
The highest component is $f_{\max} = 17$ cycles/unit, requiring a Nyquist
rate of at least 34 samples/unit - or 7140 samples over $[0, 210]$ - for
perfect reconstruction.

In [ ]:
"""nyquist_unknown.ipynb"""

# Cell 01 - Define the unknown waveform and sample it at pi

import numpy as np


# fmt: off
def f(x: float) -> float:
    """Product of four sinusoids with angular frequencies 4pi, 6pi, 10pi, 14pi.

    True spectral content (after product-to-sum expansion):
    f = 1, 11, 13, 17 cycles/unit, each with amplitude 1/8.
    The f=3 and f=7 terms cancel exactly.
    """
    return (np.sin(4 * np.pi * x) * np.sin(6 * np.pi * x)
        * np.sin(10 * np.pi * x) * np.sin(14 * np.pi * x))
# fmt: on

print(f"f(pi) = {float(f(np.pi)):.10f}")

**Sampling at 420 points (2 samples/unit).**
At 2 samples/unit the sampling rate falls below the Nyquist rate for every
frequency component.
The $f = 1$ component (the slowest) needs 2 samples/cycle, so it sits
exactly at the Nyquist boundary and aliases to DC (zero).
The $f = 11$, $13$, and $17$ components alias to much lower frequencies.
The resulting waveform looks nothing like the original.

In [ ]:
# Cell 02 - Define the plotting function and show the waveform at 420 samples

import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator


def plot_unknown_wave(n: int) -> None:
    """Plot f(x) sampled at n points over [0, 210]."""
    a, b = 0, 210
    x = np.linspace(a, b, n + 1)
    y = f(x)
    plt.figure(f"nyquist_unknown_n{n}", figsize=(10, 4))
    # fmt: off
    plt.plot(x, y, color="blue", marker="o", markersize=2,
        markerfacecolor="red", markeredgecolor="red")
    # fmt: on
    plt.title(f"Unknown Waveform ({n:,} samples, {n / b:.1f} samples/unit)")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.ylim(-1.1, 1.2)
    plt.axhline(y=0, color="lightgray")
    plt.grid(True, alpha=0.3)
    ax = plt.gca()
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    plt.tight_layout()
    plt.show()


# 2 samples/unit - all components aliased
plot_unknown_wave(420)

**Sampling at 2940 points (14 samples/unit).**
At 14 samples/unit the $f = 1$ component now receives 14 samples/cycle -
well above its Nyquist rate of 2 samples/unit - and is reconstructed correctly.
However the three higher components still alias:
$f = 11$ needs 22 samples/unit, $f = 13$ needs 26, and $f = 17$ needs 34.
The number 2940 = $14 \times 210$ is a clean integer multiple of the domain
length, which gives the aliased higher components a regular phase relationship
and produces a structured (though incorrect) pattern.

In [ ]:
# Cell 03 - 14 samples/unit: f=1 correctly reconstructed, f=11,13,17 aliased
# 2940 = 14 * 210 - clean multiple gives structured aliasing pattern

plot_unknown_wave(2940)

**Sampling at 2939 points (13.995 samples/unit).**
Reducing by just one sample breaks the clean integer relationship.
The aliased higher-frequency components now accumulate a slight phase
error across the domain, visibly distorting the pattern compared to the
2940-sample plot.
This mirrors the n=15 vs n=16 contrast in `nyquist_known.ipynb`: a single
sample can make the difference between a structured alias and a drifting one.

In [ ]:
# Cell 04 - 2939 samples: breaking the clean 14x multiple by one sample

plot_unknown_wave(2939)